# 09 Preprocess and Resample all files

This notebook, and the accompanying .py file which can be run on a high performance cluster, precprocesses and resamples all of the bold files for all of the subjects, and stores the results in a directory "preprocessed_and_resampled_data". This is a demanding task and was run for a few subjects at a time on the Cambridge High Performance Cluster (HPC).

In [1]:
import os
import pandas as pd
import numpy as np
import nilearn
import nibabel as nb

from nilearn.datasets import load_mni152_template
from nilearn.image import resample_to_img, load_img
from nilearn import plotting

from nilearn.plotting import plot_design_matrix
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn import image, masking, maskers

# 2023-02-03: changed this SIMEXP's load confounds has moved
from nilearn.interfaces.fmriprep import load_confounds
from load_confounds import Confounds

template = load_mni152_template()

/home/ca541/.conda/envs/sbp_env/lib/python3.7/site-packages/ipykernel_launcher.py:18: FutureWarning: Default resolution of the MNI template will change from 2mm to 1mm in version 0.10.0


In [2]:
def load_subject_sessions(subjects_to_not_consider = []):
    """
    Load subjects rejecting those that are in the list subjects_to_not_consider
    
    Parameters
    ----------
    subjects_to_not_consider : List
        e.g. ['099', '104', '121'], returned from just_check_output_folder()
    
    Returns
    -------
    subject_sessions : List
        e.g. ['output/sub-001/ses-visit1', 'output/sub-001/ses-visit2']
    """
    parent_dir = "output/"
    
    subject_sessions = []
    for subject_dir in sorted(os.listdir(parent_dir)):
        subject_dir_path = os.path.join(parent_dir, subject_dir)
        if os.path.isdir(subject_dir_path) and "sub" in subject_dir_path: # check for sub
            for visit_dir in sorted(os.listdir(subject_dir_path)):
                visit_dir_path = os.path.join(subject_dir_path, visit_dir)
                if os.path.isdir(visit_dir_path) and "ses" in visit_dir_path: # check for ses
                    if not any(subject_id in visit_dir_path for subject_id in subjects_to_not_consider):
                        subject_sessions.append(visit_dir_path)
    return subject_sessions #doesn't check whether all of the files are in there

def load_subject_files(subject_sessions):
    """
    Create dataframe for the subject and sessions for easy access of all files
    
    Parameters
    ----------
    subject_sessions : List
        e.g. ['output/sub-001/ses-visit1', 'output/sub-001/ses-visit2'], returned from load_subject_sessions()
        
    Returns
    -------
    df : pd.DataFrame()
    """
    df = pd.DataFrame()
    for session_dir in sorted(subject_sessions):
        dict_for_subject = {"subject": str(session_dir.split("/")[1][4:]), "session": str(session_dir.split("/")[2][-1])}
        try:
            for file in sorted(os.listdir(session_dir + "/func/")):
                if "confounds" in file:
                    complete_file_path = os.path.join(session_dir + "/func/", file)
                    if "mv" in file in file:
                        dict_for_subject["mv_confounds"] = complete_file_path
                    elif "sp_run-01" in file:
                        dict_for_subject["sp_run_01_confounds"] = complete_file_path
                    elif "sp_run-02" in file:
                        dict_for_subject["sp_run_02_confounds"] = complete_file_path
                    elif "sv" in file:
                        dict_for_subject["sv_confounds"] = complete_file_path
                    else:
                        pass
                    
                if "nii.gz" in file:
                    complete_file_path = os.path.join(session_dir + "/func/", file)
                    if "mv" in file and "brain_mask" in file:
                        dict_for_subject["mv_brain_mask"] = complete_file_path
                    elif "sp_run-01" in file and "brain_mask" in file:
                        dict_for_subject["sp_run_01_brain_mask"] = complete_file_path
                    elif "sp_run-02" in file and "brain_mask" in file:
                        dict_for_subject["sp_run_02_brain_mask"] = complete_file_path
                    elif "sv" in file and "brain_mask" in file:
                        dict_for_subject["sv_brain_mask"] = complete_file_path
                    elif "mv" in file and "desc-preproc_bold" in file:
                        dict_for_subject["mv_bold"] = complete_file_path
                    elif "sp_run-01" in file and "desc-preproc_bold" in file:
                        dict_for_subject["sp_run_01_bold"] = complete_file_path
                    elif "sp_run-02" in file and "desc-preproc_bold" in file:
                        dict_for_subject["sp_run_02_bold"] = complete_file_path
                    elif "sv" in file and "desc-preproc_bold" in file:
                        dict_for_subject["sv_bold"] = complete_file_path
                    else:
                        pass
        except Exception as e:
            print(e)
            continue
    
        new_row = pd.Series(dict_for_subject)
        df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
    return df

In [3]:
subject_sessions = load_subject_sessions()
subject_files_df = load_subject_files(subject_sessions)
subject_files_df

,subject,session,mv_confounds,mv_brain_mask,mv_bold,sp_run_01_confounds,sp_run_01_brain_mask,sp_run_01_bold,sp_run_02_confounds,sp_run_02_brain_mask,sp_run_02_bold,sv_confounds,sv_brain_mask,sv_bold
0,001,1,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...
1,001,2,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...
2,001,4,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...
3,001,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,output/sub-001/ses-visit5/func/sub-001_ses-vis...,output/sub-001/ses-visit5/func/sub-001_ses-vis...,output/sub-001/ses-visit5/func/sub-001_ses-vis...
4,002,1,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,120,3,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,NaN,NaN,NaN,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...
402,120,4,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,NaN,NaN,NaN,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...
403,121,1,output/sub-121/ses-visit1/func/sub-121_ses-vis...,output/sub-121/ses-visit1/func/sub-121_ses-vis...,output/sub-121/ses-visit1/func/sub-121_ses-vis...,output/sub-

In [4]:
subject_files_df.isna().sum()

subject                   0
session                   0
mv_confounds             65
mv_brain_mask            65
mv_bold                  65
sp_run_01_confounds       6
sp_run_01_brain_mask      6
sp_run_01_bold            6
sp_run_02_confounds     176
sp_run_02_brain_mask    176
sp_run_02_bold          176
sv_confounds             11
sv_brain_mask            24
sv_bold                  24
dtype: int64

In [5]:
subject_files_df.loc[0]["mv_brain_mask"]

'output/sub-001/ses-visit1/func/sub-001_ses-visit1_task-mv_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz'

In [6]:
def check_nii_shapes(subject_files_df):
    stim_on_files = subject_files_df["sp_run_01_bold"].tolist()
    subjects_to_not_consider = []
    
    for file in stim_on_files:
        try:
            loaded_data = nb.load(file)
            if loaded_data.shape != (57, 68, 65, 244):
                print(file, loaded_data.shape)
                subjects_to_not_consider.append(file.split("/")[1].split("-")[1])
        except:
            print("didn't happen for", file)
            
    return list(set(subjects_to_not_consider))

In [7]:
check_nii_shapes(subject_files_df)

didn't happen for nan
didn't happen for nan
didn't happen for nan
output/sub-035/ses-visit4/func/sub-035_ses-visit4_task-sp_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz (57, 68, 65, 248)
didn't happen for nan
didn't happen for nan
output/sub-060/ses-visit1/func/sub-060_ses-visit1_task-sp_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz (57, 68, 58, 244)
didn't happen for nan
output/sub-098/ses-visit4/func/sub-098_ses-visit4_task-sp_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz (57, 68, 60, 244)


['035', '060', '098']

In [ ]:
def preprocess_file(subject_id = "097", task = "sp_run_01", session = "1"):
    bold_file = subject_files_df.loc[(subject_files_df.subject == subject_id) & (subject_files_df.session == session)][task + "_bold"].tolist()[0]
    mask_file = subject_files_df.loc[(subject_files_df.subject == subject_id) & (subject_files_df.session == session)][task + "_brain_mask"].tolist()[0]
    
    # got code from P9 here: https://github.com/SIMEXP/load_confounds/pull/51/commits/26467de1387ad29773ff394dcb62f40c03ebb088
    # added "high_pass" in strategy and n_motion doesn't exist
    confounds = load_confounds(bold_file,
                              strategy=["high_pass", "motion", "wm_csf", "global_signal"], 
                              motion="basic", 
                              # n_motion = 0,
                              wm_csf="basic",
                              global_signal="basic")
    # print("confounds shape:", confounds)
    # print(len(confounds), type(confounds[0]), confounds[0], type(confounds[1]), confounds[1], confounds[1].shape)
    # display(confounds[0])
    # print(confounds[1])
    t_r = 2.5
    high_pass= 0.006
    low_pass = None
    bold_img = nilearn.image.load_img(bold_file)
    bold_img = bold_img.slicer[:,:,:,4:]
    #plotting.plot_epi(bold_img.slicer[:, :, :, 50])
    confounds_matrix = confounds[0].iloc[4:]
    cleaned_bold = nilearn.image.clean_img(bold_img, 
                                              confounds=confounds_matrix, 
                                              detrend=False, 
                                              standardize=False, 
                                              low_pass=low_pass, 
                                              high_pass=high_pass,
                                              ensure_finite=True,
                                              mask_img=mask_file,
                                              t_r = t_r
                                             )
    cleaned_bold = nilearn.image.smooth_img(cleaned_bold, fwhm=7)
    print("cleaned_bold shape:", cleaned_bold.shape)
    # plotting.plot_epi(cleaned_bold.slicer[:, :, :, 50])
    return cleaned_bold, mask_file

def preprocess_all_files(subject_files_df):
    output_dir = "preprocessed_and_resampled_data/"
    try:
        os.mkdir(output_dir)
    except:
        pass
    
    for index, row in subject_files_df.iterrows():
        print("Preprocessing subject:", row["subject"], "session:", row["session"])
        try:
            preprocessed_and_resampled_path = os.path.join(output_dir, "sub-" + row["subject"], "ses-visit" + row["session"], "sp_run_01", "cleaned_file.nii.gz")
            try:
                os.makedirs(os.path.dirname(preprocessed_and_resampled_path))
            except:
                pass
            
            if os.path.exists(preprocessed_and_resampled_path):
                print(preprocessed_and_resampled_path, "already exists!!!")
                subject_files_df.at[index,'resampled_and_preprocessed_bold'] = preprocessed_and_resampled_path
            else:
                cleaned_bold, mask_file = preprocess_file(subject_id = row["subject"], task = "sp_run_01", session = row["session"])
                resampled = resample_to_img(cleaned_bold, template)
                print("resampled shape:", resampled.shape, "saved at:", preprocessed_and_resampled_path)
                resampled.to_filename(preprocessed_and_resampled_path)
                subject_files_df.at[index,'resampled_and_preprocessed_bold'] = preprocessed_and_resampled_path
        except:
            print("DIDN'T HAPPEN FOR subject:", row["subject"], "session:", row["session"])
    return subject_files_df

In [9]:
subject_files_df_updated = preprocess_all_files(subject_files_df)

Preprocessing subject: 001 session: 1
preprocessed_and_resampled_data/sub-001/ses-visit1/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 001 session: 2
preprocessed_and_resampled_data/sub-001/ses-visit2/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 001 session: 4
preprocessed_and_resampled_data/sub-001/ses-visit4/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 001 session: 5
DIDN'T HAPPEN FOR subject: 001 session: 5
Preprocessing subject: 002 session: 1
preprocessed_and_resampled_data/sub-002/ses-visit1/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 002 session: 2
preprocessed_and_resampled_data/sub-002/ses-visit2/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 002 session: 3
preprocessed_and_resampled_data/sub-002/ses-visit3/sp_run_01/cleaned_file.nii.gz already exists!!!
Preprocessing subject: 002 session: 4
preprocessed_and_resampled_data/sub-002/ses-visit4/sp_run_01

In [10]:
subject_files_df_updated.to_excel("subject_files_feb_2023.xlsx")

In [11]:
subject_files_df_updated

,subject,session,mv_confounds,mv_brain_mask,mv_bold,sp_run_01_confounds,sp_run_01_brain_mask,sp_run_01_bold,sp_run_02_confounds,sp_run_02_brain_mask,sp_run_02_bold,sv_confounds,sv_brain_mask,sv_bold,resampled_and_preprocessed_bold
0,001,1,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,output/sub-001/ses-visit1/func/sub-001_ses-vis...,preprocessed_and_resampled_data/sub-001/ses-vi...
1,001,2,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,output/sub-001/ses-visit2/func/sub-001_ses-vis...,preprocessed_and_resampled_data/sub-001/ses-vi...
2,001,4,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,output/sub-001/ses-visit4/func/sub-001_ses-vis...,preprocessed_and_resampled_data/sub-001/ses-vi...
3,001,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,output/sub-001/ses-visit5/func/sub-001_ses-vis...,output/sub-001/ses-visit5/func/sub-001_ses-vis...,output/sub-001/ses-visit5/func/sub-001_ses-vis...,NaN
4,002,1,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,output/sub-002/ses-visit1/func/sub-002_ses-vis...,preprocessed_and_resampled_data/sub-002/ses-vi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,120,3,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,NaN,NaN,NaN,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,output/sub-120/ses-visit3/func/sub-120_ses-vis...,preprocessed_and_resampled_data/sub-120/ses-vi...
402,120,4,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,output/sub-120/ses-visit4/func/sub-120_ses-vis...,NaN,NaN,NaN,output/sub-120/ses-visit4/func/